In [2]:
"""
Step 1: Aggregate age-level PM2.5-attributable patient data into
city x disease x year totals (collapsing over the 'age' dimension).

INPUT ASSUMPTION (based on the screenshot you shared):
    A table with columns:
        air_scenario, disease, city, year, age,
        cross_patient_total, mo_total_min, mo_total_max
    stored either as:
        (a) one CSV/XLSX per (air_scenario, year) combo, e.g. "earlypeak_NZ_CL_2030.xlsx", OR
        (b) one big long-format file covering all years/scenarios.

    ==> Edit INPUT_MODE and the corresponding paths below to match your actual files.

OUTPUT:
    One CSV per (air_scenario, year) with columns:
        city, disease, cross_patient_total_sum, mo_total_sum, local_mo_total
    saved to OUT_DIR, following the same naming convention as your existing
    CITYSUM_DIR files but WITH a disease column preserved (not collapsed).

    File name: citysum_bydisease_{air}_{year}.csv
"""

import pandas as pd
import numpy as np
import os, glob

# ============ CONFIG — EDIT THESE TO MATCH YOUR ACTUAL FILES ============

# Mode 'multi_file': one file per (air_scenario, year), e.g. your screenshot workbook
# Mode 'single_file': one long file with all years/scenarios stacked
INPUT_MODE = "multi_file"   # or "single_file"

# --- multi_file mode settings ---
RAW_DIR = "/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5"  # EDIT: folder containing raw files
RAW_FILE_TPL = "{air}_{year}.csv"   # EDIT: filename pattern, e.g. "earlypeak_NZ_CL_2030.csv"

# --- single_file mode settings ---
SINGLE_FILE_PATH = "/Volumes/UCL/论文工作/空气污染/health_burden/all_age_level_raw.csv"

air_scenarios = ['earlypeak_NZ_CL']
years = list(range(2020, 2061, 10))

OUT_DIR = "/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/citysum_bydisease"
os.makedirs(OUT_DIR, exist_ok=True)

# Column names in the raw table (edit if your actual column names differ)
COL_AIR      = "air_scenario"
COL_DISEASE  = "disease"
COL_CITY     = "city"
COL_YEAR     = "year"
COL_AGE      = "age"
COL_CPT      = "cross_patient_total"
COL_MO       = "mo_total"

# ============ HELPER ============

def read_any(path):
    """Read a csv or xlsx file into a DataFrame, trying all sheets if xlsx."""
    if path.lower().endswith(".csv"):
        return pd.read_csv(path)
    else:
        xls = pd.ExcelFile(path)
        dfs = [pd.read_excel(path, sheet_name=sh) for sh in xls.sheet_names]
        return pd.concat(dfs, ignore_index=True)

def clean_cols(df):
    df.columns = [c.strip() if isinstance(c, str) else c for c in df.columns]
    return df

# ============ MAIN ============

if INPUT_MODE == "single_file":
    print(f"Reading single long-format file: {SINGLE_FILE_PATH}")
    big_df = clean_cols(read_any(SINGLE_FILE_PATH))
    big_df[COL_CITY] = big_df[COL_CITY].astype(str).str.strip()
    big_df[COL_DISEASE] = big_df[COL_DISEASE].astype(str).str.strip()

for air in air_scenarios:
    for year in years:

        if INPUT_MODE == "multi_file":
            raw_path = os.path.join(RAW_DIR, RAW_FILE_TPL.format(air=air, year=year))
            matches = glob.glob(raw_path)
            if not matches:
                print(f"⚠️ Raw file not found: {raw_path}")
                continue
            df = clean_cols(read_any(matches[0]))
            df[COL_CITY] = df[COL_CITY].astype(str).str.strip()
            df[COL_DISEASE] = df[COL_DISEASE].astype(str).str.strip()
        else:
            df = big_df[(big_df[COL_AIR] == air) & (big_df[COL_YEAR] == year)].copy()
            if df.empty:
                print(f"⚠️ No rows found for air={air}, year={year}")
                continue

        # --- Collapse over age: sum cross_patient_total and mo_total
        #     within each (city, disease) group ---
        grouped = (
            df.groupby([COL_CITY, COL_DISEASE], as_index=False)
              .agg(
                  cross_patient_total_sum=(COL_CPT, "sum"),
                  mo_total_sum=(COL_MO, "sum"),
              )
        )

        grouped["local_mo_total"] = grouped["mo_total_sum"] - grouped["cross_patient_total_sum"]

        grouped.insert(0, "year", year)
        grouped.insert(0, "air_scenario", air)

        out_path = os.path.join(OUT_DIR, f"citysum_bydisease_{air}_{year}.csv")
        grouped.to_csv(out_path, index=False, float_format="%.8g")
        print(f"✅ Saved: {out_path} | {grouped[COL_DISEASE].nunique()} diseases, "
              f"{grouped[COL_CITY].nunique()} cities, {len(grouped)} rows")

print("\nDone. Each output file has columns:")
print("  air_scenario, year, city, disease, cross_patient_total_sum, mo_total_sum, local_mo_total")

✅ Saved: /Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/citysum_bydisease/citysum_bydisease_earlypeak_NZ_CL_2020.csv | 5 diseases, 360 cities, 1800 rows
✅ Saved: /Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/citysum_bydisease/citysum_bydisease_earlypeak_NZ_CL_2030.csv | 5 diseases, 360 cities, 1800 rows
✅ Saved: /Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/citysum_bydisease/citysum_bydisease_earlypeak_NZ_CL_2040.csv | 5 diseases, 360 cities, 1800 rows
✅ Saved: /Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/citysum_bydisease/citysum_bydisease_earlypeak_NZ_CL_2050.csv | 5 diseases, 360 cities, 1800 rows
✅ Saved: /Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/citysum_bydisease/citysum_bydisease_earlypeak_NZ_CL_2060.csv | 5 diseases, 360 cities, 1800 rows

Done. Each output file has columns:
  air_scenario, year, city, disease, cross_patient_total_sum, mo_total_sum, local_mo_total


In [4]:
"""
Step 2 (simplified): You already have the a_ij row-probability matrices
computed and saved to disk. This script SKIPS the distance/GDP/hospital-resource
computation entirely, and just:

  1. Reads the existing a_ij matrix for each (air_scenario, year).
  2. Reads the per-disease city totals from Step 1
     (citysum_bydisease_{air}_{year}.csv).
  3. For each disease, takes cross_patient_total_sum as the source vector V,
     aligns it to the a_ij matrix's row (source-city) index,
     and computes F = a_ij * V (row-wise multiplication).
  4. Saves one flow matrix per disease per (air, year).

ASSUMPTION (based on your file name "aij_rowprob_avg_earlypeak_NZ_CL_2040"):
    The a_ij matrix is already averaged over fer/rcp scenarios, and only
    varies by (air_scenario, year) — i.e. one file per (air, year), NOT
    per (air, fer, rcp, year). If this is wrong (you actually have separate
    a_ij files per fer/rcp too), tell me and I'll add that loop back in.
"""

import pandas as pd
import numpy as np
import os, glob

# ============ CONFIG — EDIT THESE ============

AIJ_DIR = "/Volumes/UCL/论文工作/空气污染/cross_flow_truncated/averaged_results/aij_avg"  # a_ij文件实际所在目录
AIJ_TPL = "aij_rowprob_avg_{air}_{year}.csv"  # 与实际文件名一致，无需再改

CITYSUM_BYDISEASE_DIR = "/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/citysum_bydisease"
CITYSUM_BYDISEASE_TPL = "citysum_bydisease_{air}_{year}.csv"

OUT_DIR = "/Volumes/UCL/论文工作/空气污染/cross_flow_by_disease"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "flow_results"), exist_ok=True)

air_scenarios = ['earlypeak_NZ_CL']
years = list(range(2020, 2061, 10))

COL_CITY    = "city"
COL_DISEASE = "disease"
COL_CPT     = "cross_patient_total_sum"   # source strength to redistribute

# ============ HELPER ============

def read_aij(path):
    """a_ij file: first column = source city (index), remaining columns = target cities."""
    if path.lower().endswith(".csv"):
        df = pd.read_csv(path, index_col=0)
    else:
        df = pd.read_excel(path, index_col=0)
    df.index = df.index.astype(str).str.strip()
    df.columns = [str(c).strip() for c in df.columns]
    return df.astype(float)

# ============ MAIN ============

for air in air_scenarios:
    for year in years:

        # --- Load a_ij matrix (already computed) ---
        aij_path = os.path.join(AIJ_DIR, AIJ_TPL.format(air=air, year=year))
        matches = glob.glob(aij_path)
        if not matches:
            print(f"⚠️ a_ij file not found: {aij_path}")
            continue
        A_rowprob = read_aij(matches[0])
        rows = A_rowprob.index.tolist()  # source cities

        # --- Load per-disease city totals from Step 1 ---
        v_path = os.path.join(CITYSUM_BYDISEASE_DIR, CITYSUM_BYDISEASE_TPL.format(air=air, year=year))
        if not os.path.exists(v_path):
            print(f"⚠️ Per-disease V file not found: {v_path}")
            continue
        Vdf_all = pd.read_csv(v_path)
        Vdf_all[COL_CITY] = Vdf_all[COL_CITY].astype(str).str.strip()
        Vdf_all[COL_DISEASE] = Vdf_all[COL_DISEASE].astype(str).str.strip()

        diseases = Vdf_all[COL_DISEASE].unique().tolist()
        print(f"\n=== {air} | {year} | a_ij shape={A_rowprob.shape} | {len(diseases)} diseases ===")

        for disease in diseases:
            Vdf = Vdf_all[Vdf_all[COL_DISEASE] == disease].copy()

            V_vec = (
                Vdf.set_index(COL_CITY)[COL_CPT]
                .astype(float)
                .reindex(rows)      # align to a_ij row order; missing cities -> NaN
                .fillna(0.0)
            )

            # Row-wise multiply: F_ij = a_ij * V_i
            F = A_rowprob.mul(V_vec, axis=0)

            disease_tag = disease.strip().replace(" ", "_").replace("/", "-")
            out_path = os.path.join(
                OUT_DIR, f"flow_results/flow_patientnum_{disease_tag}_{air}_{year}.csv"
            )
            F.to_csv(out_path, float_format="%.8g")
            print(f"   ✅ {disease}: flow matrix saved -> {out_path} | shape={F.shape} | total={F.values.sum():.1f}")

print("\nDone.")


=== earlypeak_NZ_CL | 2020 | a_ij shape=(360, 165) | 5 diseases ===
   ✅ Chronic Obstructive Pulmonary Disease: flow matrix saved -> /Volumes/UCL/论文工作/空气污染/cross_flow_by_disease/flow_results/flow_patientnum_Chronic_Obstructive_Pulmonary_Disease_earlypeak_NZ_CL_2020.csv | shape=(360, 165) | total=70829.1
   ✅ Ischaemic Heart Disease: flow matrix saved -> /Volumes/UCL/论文工作/空气污染/cross_flow_by_disease/flow_results/flow_patientnum_Ischaemic_Heart_Disease_earlypeak_NZ_CL_2020.csv | shape=(360, 165) | total=117897.3
   ✅ Lower Respiratory Infections: flow matrix saved -> /Volumes/UCL/论文工作/空气污染/cross_flow_by_disease/flow_results/flow_patientnum_Lower_Respiratory_Infections_earlypeak_NZ_CL_2020.csv | shape=(360, 165) | total=19773.4
   ✅ Lung Cancer: flow matrix saved -> /Volumes/UCL/论文工作/空气污染/cross_flow_by_disease/flow_results/flow_patientnum_Lung_Cancer_earlypeak_NZ_CL_2020.csv | shape=(360, 165) | total=15755.0
   ✅ Strokes: flow matrix saved -> /Volumes/UCL/论文工作/空气污染/cross_flow_by_disease/

In [1]:
# =============================================================================
# Step 3: Per-disease flow maps (Fig2 panel "a" style — main map + zoom insets)
#
# This reuses the map-drawing logic from your Fig2 script, but:
#   - load_matrix() now takes a `disease` argument and reads from the
#     per-disease flow files produced in Step 2
#     (flow_patientnum_{disease_tag}_{air}_{year}.csv)
#   - The income/sankey/dot-plot panels (old panel b) are removed entirely —
#     only the flow-map + zoom-inset layout (old panel a) is kept
#   - The whole map block is now wrapped in a loop over diseases, saving
#     ONE output file per disease: Fig_flowmap_{disease}_{year0}_{year1}.png
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle, ConnectionPatch
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
from shapely.geometry import box as shapely_box
from pyproj import Transformer

# ── 0. Unified font sizes ─────────────────────────────────────────────────────
FS_BASE        = 7
FS_TITLE       = 8
FS_PANEL_LABEL = 10
FS_SCALE       = 6.5

plt.rcParams.update({
    "font.family":      "Arial",
    "font.size":        FS_BASE,
    "axes.titlesize":   FS_TITLE,
    "axes.titleweight": "bold",
    "axes.titlepad":    3,
})

# ── 1. Paths ──────────────────────────────────────────────────────────────────
# NEW: per-disease flow files, output of Step 2
DISEASE_FLOW_DIR = Path("/Volumes/UCL/论文工作/空气污染/cross_flow_by_disease/flow_results")
DISEASE_FLOW_TPL  = "flow_patientnum_{disease_tag}_{air}_{year}.csv"

# Used only to discover the list of diseases (Step 1 output, any year works)
CITYSUM_BYDISEASE_DIR = Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/citysum_bydisease")
CITYSUM_BYDISEASE_TPL = "citysum_bydisease_{air}_{year}.csv"
DISEASE_LIST_YEAR     = 2020  # any year is fine, just to enumerate disease names

SHP_PATH   = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/city_shp/shi_en.shp")
CHINA_SHP  = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国底图-中图社审过版本/中国底图/中国面.shp")
CHINA_SHP2 = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国国界线/九段线/九段线和群岛.shp")

OUT_DIR = Path("/Users/shirley/Desktop/plots_V2/flowmaps_by_disease")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIO      = "earlypeak_NZ_CL"
PROJ_STR      = "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105"
COMPARE_YEARS = [(2020, "a"), (2060, "b")]   # same two-year comparison as before

# ── 2. Flow color scale ───────────────────────────────────────────────────────
FLOW_CMAP = plt.cm.plasma_r
FLOW_VMIN, FLOW_VMAX = 1, 2500
FLOW_NORM = mcolors.LogNorm(vmin=FLOW_VMIN, vmax=FLOW_VMAX)

# ── 3. Zoom regions ───────────────────────────────────────────────────────────
ZOOM_REGIONS = {
    "Beijing-Tianjin-Hebei": (113.5, 119.5, 36.0, 41.0),
    "Chengdu-Chongqing":     (102.5, 108.0, 28.0, 32.0),
    "Yangtze River Delta":   (118.5, 122.5, 29.5, 33.0),
    "Pearl River Delta":     (112.0, 115.5, 21.5, 24.5),
    "Urumqi":                (84.5,  90.5,  41.5, 46.0),
}
MAIN_MAP_BBOX = (80, 139, 10, 55)
ZOOM_ASPECT   = 1.4

# ── 4. Spatial data (identical to your original script) ──────────────────────
china_border = gpd.read_file(CHINA_SHP).to_crs(PROJ_STR)
jiudanline   = gpd.read_file(CHINA_SHP2).to_crs(PROJ_STR)
city_shp     = gpd.read_file(SHP_PATH).to_crs(PROJ_STR)
city_shp["geometry"]  = city_shp.geometry.representative_point()
city_shp["cx"]        = city_shp.geometry.x
city_shp["cy"]        = city_shp.geometry.y
city_shp["city_name"] = city_shp["English"].str.strip()
city_pts = (city_shp.drop_duplicates(subset="city_name", keep="first")
            .set_index("city_name")[["cx", "cy"]].to_dict("index"))

_tf = Transformer.from_crs("EPSG:4326", PROJ_STR, always_xy=True)
_HHY_X, _HHY_Y = _tf.transform([127.5, 98.5], [50.2, 25.0])
_NANHAI_BOUNDS = (gpd.GeoDataFrame(geometry=[shapely_box(105, 2, 122, 24)], crs="EPSG:4326")
                  .to_crs(PROJ_STR).total_bounds)

xmin, ymin, xmax, ymax = china_border.total_bounds
pad_x = (xmax - xmin) * 0.01
pad_y = (ymax - ymin) * 0.01

# ── 5. Geometry helpers ───────────────────────────────────────────────────────
def lonlat_bbox_to_proj(lon_min, lon_max, lat_min, lat_max):
    xs, ys = _tf.transform([lon_min, lon_max], [lat_min, lat_max])
    return min(xs), min(ys), max(xs), max(ys)

def pad_bbox_to_aspect(bbox, ratio):
    x0, y0, x1, y1 = bbox
    cx, cy = (x0+x1)/2, (y0+y1)/2
    w, h = x1-x0, y1-y0
    if w/h < ratio:
        x0, x1 = cx - h*ratio/2, cx + h*ratio/2
    else:
        y0, y1 = cy - w/ratio/2, cy + w/ratio/2
    return x0, y0, x1, y1

def disease_to_tag(disease):
    return disease.strip().replace(" ", "_").replace("/", "-")

# ── 6. Matrix loader — NOW PER-DISEASE ────────────────────────────────────────
def load_matrix(year, disease_tag):
    """Load the disease-specific OD flow matrix for a given year."""
    path = DISEASE_FLOW_DIR / DISEASE_FLOW_TPL.format(
        disease_tag=disease_tag, air=SCENARIO, year=year
    )
    if not path.exists():
        print(f"⚠️ Missing flow file: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path, index_col=0)
    df.index   = df.index.str.strip()
    df.columns = df.columns.str.strip()
    df = df[df.index.notna()]
    df = df.loc[~df.index.isin(["total"]), ~df.columns.isin(["total"])]
    np.fill_diagonal(df.values, 0)
    valid  = set(city_pts.keys())
    cities = [c for c in df.index if c in valid]
    return df.loc[cities, [c for c in df.columns if c in valid]]

def compute_all_edges(df):
    edges = []
    if df.empty:
        return pd.DataFrame(edges)
    for ci in df.index:
        for cj in df.columns:
            if ci == cj: continue
            try:
                flow = float(df.loc[ci, cj])
            except KeyError:
                continue
            if np.isnan(flow) or flow <= 1: continue
            edges.append({"ori": ci, "dst": cj, "flow": flow})
    return pd.DataFrame(edges)

# ── 7. Arrow drawing (unchanged) ──────────────────────────────────────────────
def draw_arrows_on_ax(ax, edge_df, lw_scale=1.0, clip_bbox=None):
    if edge_df.empty: return
    plot_df = edge_df.copy()
    if clip_bbox is not None:
        x0, y0, x1, y1 = clip_bbox
        def _in(row):
            pt = city_pts.get(row["dst"])
            return pt is not None and x0<=pt["cx"]<=x1 and y0<=pt["cy"]<=y1
        plot_df = plot_df[plot_df.apply(_in, axis=1)]
        if plot_df.empty: return
    plot_df = plot_df.sort_values("flow")
    for _, row in plot_df.iterrows():
        try:
            ox=city_pts[row["ori"]]["cx"]; oy=city_pts[row["ori"]]["cy"]
            dx=city_pts[row["dst"]]["cx"]; dy=city_pts[row["dst"]]["cy"]
        except KeyError: continue
        flow  = row["flow"]
        color = FLOW_CMAP(FLOW_NORM(np.clip(flow, FLOW_VMIN, FLOW_VMAX)))
        t     = np.log10(np.clip(flow, FLOW_VMIN, FLOW_VMAX)) / np.log10(FLOW_VMAX)
        ax.add_patch(mpatches.FancyArrowPatch(
            (ox, oy), (dx, dy), connectionstyle="arc3,rad=0.15",
            arrowstyle="-|>" if flow >= 10 else "-",
            mutation_scale=(3+5*t)*lw_scale,
            linewidth=(0.15+0.9*t)*lw_scale,
            color=color, alpha=0.12+0.75*t,
            shrinkA=0, shrinkB=1, zorder=3+t, clip_on=True))

# ── 8. Map functions (unchanged apart from disease-aware title) ──────────────
def _add_scale_bar(ax):
    ax.add_artist(AnchoredSizeBar(
        ax.transData, 500_000, "500 km", loc="upper left",
        pad=0.3, borderpad=0.5, sep=3, color="black", frameon=False,
        size_vertical=(ymax-ymin)*0.003,
        fontproperties=fm.FontProperties(size=FS_SCALE, weight="bold")))

def draw_flow_map(ax_main, year, disease_tag, tag, total_n=None):
    df      = load_matrix(year, disease_tag)
    edge_df = compute_all_edges(df)
    china_border.plot(ax=ax_main, facecolor="#eef1f4", edgecolor="black", linewidth=0.3)
    city_shp.plot(ax=ax_main, facecolor="none", edgecolor="#b8bfc7", linewidth=0.15, zorder=1)
    jiudanline.plot(ax=ax_main, facecolor="white", edgecolor="black", linewidth=0.25, alpha=0.4)
    draw_arrows_on_ax(ax_main, edge_df)
    ax_main.plot(_HHY_X, _HHY_Y, color="black", lw=0.6, linestyle="--", dashes=(4,3), zorder=5)
    if MAIN_MAP_BBOX is not None:
        bx0,by0,bx1,by1 = lonlat_bbox_to_proj(*MAIN_MAP_BBOX)
        ax_main.set_xlim(bx0,bx1); ax_main.set_ylim(by0,by1)
    else:
        ax_main.set_xlim(xmin-pad_x, xmax+pad_x)
        ax_main.set_ylim(ymin+(ymax-ymin)*0.05, ymax+pad_y)
    _x0,_x1 = ax_main.get_xlim(); _y0,_y1 = ax_main.get_ylim()
    ax_main.set_aspect("equal")
    ax_main.set_box_aspect((_y1-_y0)/(_x1-_x0))
    ax_main.set_axis_off()
    ax_main.text(-0.02, 1.01, tag, transform=ax_main.transAxes,
                 fontsize=FS_PANEL_LABEL, fontweight="bold", va="bottom")
    title_str = str(year)
    if total_n is not None:
        title_str += f"  (n={total_n:,.0f})"
    ax_main.set_title(title_str, fontsize=FS_TITLE, fontweight="bold", pad=4)
    _add_scale_bar(ax_main)
    return edge_df

def draw_zoom_panel(ax, edge_df, bbox_proj, title):
    x0,y0,x1,y1 = bbox_proj
    china_border.plot(ax=ax, facecolor="#eef1f4", edgecolor="black", linewidth=0.3)
    city_shp.plot(ax=ax, facecolor="none", edgecolor="#b8bfc7", linewidth=0.2, zorder=1)
    draw_arrows_on_ax(ax, edge_df, lw_scale=1.3, clip_bbox=bbox_proj)
    ax.set_xlim(x0,x1); ax.set_ylim(y0,y1)
    ax.set_aspect("equal")
    ax.set_box_aspect((y1-y0)/(x1-x0))
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_edgecolor("black"); sp.set_linewidth(0.8)
    ax.set_title(title, fontsize=6, fontweight="normal", pad=1, loc="center")

def draw_zoom_rect_and_connector(fig, ax_main, ax_zoom, bbox_proj):
    x0,y0,x1,y1 = bbox_proj
    ax_main.add_patch(Rectangle((x0,y0),x1-x0,y1-y0,
                                  fill=False,edgecolor="black",linewidth=0.8,zorder=8))
    fig.add_artist(ConnectionPatch(
        xyA=(x1,(y0+y1)/2), coordsA="data", axesA=ax_main,
        xyB=(0,0.5),         coordsB="axes fraction", axesB=ax_zoom,
        color="black", linewidth=0.5, zorder=1))

def add_nanhai_inset(ax_main, edge_df):
    x0,y0,x1,y1 = _NANHAI_BOUNDS
    axins = ax_main.inset_axes([0.1, 0.02, 0.22, 0.25])
    axins.set_facecolor("white")
    china_border.plot(ax=axins, facecolor="#eef1f4", edgecolor="black", linewidth=0.3)
    jiudanline.plot(ax=axins, edgecolor="black", linewidth=0.4)
    draw_arrows_on_ax(axins, edge_df, lw_scale=0.8, clip_bbox=(x0,y0,x1,y1))
    axins.set_xlim(x0,x1); axins.set_ylim(y0,y1)
    axins.set_xticks([]); axins.set_yticks([])
    for sp in axins.spines.values():
        sp.set_edgecolor("black"); sp.set_linewidth(0.6)

def add_flow_colorbar(cax):
    sm = plt.cm.ScalarMappable(cmap=FLOW_CMAP, norm=FLOW_NORM)
    sm.set_array([])
    cb = plt.colorbar(sm, cax=cax, orientation="horizontal")
    cb.set_label("Outflow patients", fontsize=FS_BASE, labelpad=3)
    cb.ax.tick_params(labelsize=FS_SCALE, length=2)
    cb.ax.xaxis.set_major_formatter(
        plt.matplotlib.ticker.FuncFormatter(lambda x, _: f"{x:g}"))
    cb.outline.set_visible(False)

# ── 9. Discover disease list from Step 1 output ───────────────────────────────
disease_list_path = CITYSUM_BYDISEASE_DIR / CITYSUM_BYDISEASE_TPL.format(
    air=SCENARIO, year=DISEASE_LIST_YEAR
)
disease_list_df = pd.read_csv(disease_list_path)
diseases = sorted(disease_list_df["disease"].astype(str).str.strip().unique().tolist())
print(f"Found {len(diseases)} diseases: {diseases}")

# ── 10. Main loop: one flow-map figure PER DISEASE ────────────────────────────
NZ  = len(ZOOM_REGIONS)
L_MAP = 0.01
PL_X, PL_Y = -0.005, 0.98

for disease in diseases:
    disease_tag = disease_to_tag(disease)
    print(f"\n=== Drawing flow map for disease: {disease} ===")

    fig = plt.figure(figsize=(180/25.4, 95/25.4), dpi=300)
    subfig = fig.subfigures(1, 1)

    gs_left = GridSpec(
        NZ+2, 2, figure=subfig,
        width_ratios=[3.4, 1],
        height_ratios=[0.3] + [1]*NZ + [0.3],
        left=L_MAP, right=0.47,
        top=0.95, bottom=0.10,
        hspace=0.32, wspace=0.2,
    )
    gs_right = GridSpec(
        NZ+2, 2, figure=subfig,
        width_ratios=[3.4, 1],
        height_ratios=[0.3] + [1]*NZ + [0.3],
        left=0.53, right=1-L_MAP,
        top=0.95, bottom=0.10,
        hspace=0.32, wspace=0.2,
    )

    any_data = False
    for block, (year, _) in enumerate(COMPARE_YEARS):
        gs = gs_left if block == 0 else gs_right
        ax_main  = subfig.add_subplot(gs[:, 0])
        ax_zooms = [subfig.add_subplot(gs[i+1, 1]) for i in range(NZ)]

        df = load_matrix(year, disease_tag)
        if df.empty:
            ax_main.set_axis_off()
            for ax_z in ax_zooms:
                ax_z.set_axis_off()
            continue
        any_data = True

        total_n = df.values.sum()
        edge_df = draw_flow_map(ax_main, year, disease_tag, tag="", total_n=total_n)
        add_nanhai_inset(ax_main, edge_df)
        for ax_z, (region_name, lonlat) in zip(ax_zooms, ZOOM_REGIONS.items()):
            raw_bbox  = lonlat_bbox_to_proj(*lonlat)
            bbox_proj = pad_bbox_to_aspect(raw_bbox, ZOOM_ASPECT)
            draw_zoom_panel(ax_z, edge_df, bbox_proj, region_name)
            draw_zoom_rect_and_connector(subfig, ax_main, ax_z, bbox_proj)

    if not any_data:
        print(f"   ⚠️ No flow data found for '{disease}' in either year — skipping save.")
        plt.close(fig)
        continue

    subfig.text(PL_X, PL_Y, disease, fontsize=FS_PANEL_LABEL,
                fontweight="bold", va="top", ha="left")

    cax = subfig.add_axes([0.13, 0.06, 1 - 2*0.13, 0.02])
    add_flow_colorbar(cax)

    out_path = OUT_DIR / f"Fig_flowmap_{disease_tag}_2020_2060.png"
    fig.savefig(out_path, dpi=400, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"   ✓ Saved → {out_path}")

print("\nAll done.")

Found 5 diseases: ['Chronic Obstructive Pulmonary Disease', 'Ischaemic Heart Disease', 'Lower Respiratory Infections', 'Lung Cancer', 'Strokes']

=== Drawing flow map for disease: Chronic Obstructive Pulmonary Disease ===
   ✓ Saved → /Users/shirley/Desktop/plots_V2/flowmaps_by_disease/Fig_flowmap_Chronic_Obstructive_Pulmonary_Disease_2020_2060.png

=== Drawing flow map for disease: Ischaemic Heart Disease ===
   ✓ Saved → /Users/shirley/Desktop/plots_V2/flowmaps_by_disease/Fig_flowmap_Ischaemic_Heart_Disease_2020_2060.png

=== Drawing flow map for disease: Lower Respiratory Infections ===
   ✓ Saved → /Users/shirley/Desktop/plots_V2/flowmaps_by_disease/Fig_flowmap_Lower_Respiratory_Infections_2020_2060.png

=== Drawing flow map for disease: Lung Cancer ===
   ✓ Saved → /Users/shirley/Desktop/plots_V2/flowmaps_by_disease/Fig_flowmap_Lung_Cancer_2020_2060.png

=== Drawing flow map for disease: Strokes ===
   ✓ Saved → /Users/shirley/Desktop/plots_V2/flowmaps_by_disease/Fig_flowmap_Strok